# ANT — A100 Unified Chat Training

**828K parameter looping transformer** with persistent external memory.

Uses **causal sliding windows** for ALL training — LM, QA, and chat.
No standard causal forward anywhere.

## 3-Phase Curriculum
1. **Phase 1** — English LM (Wikipedia + shell)
2. **Phase 2** — LM + Memory QA (bAbI with cross-attention)
3. **Phase 3** — LM + QA + Chat (tagged conversational pairs)

## A100 Optimizations
- **BF16 mixed precision** (A100 Tensor Cores)
- **Large batch sizes** (B=256, saturate GPU)
- **torch.compile** (inductor kernel fusion)
- **Checkpoints to HuggingFace Hub**

## 1. Setup

In [ ]:
# Verify GPU and install dependencies
!nvidia-smi
!pip install -q datasets numpy torch huggingface_hub

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Clone repo and login to HuggingFace
import os

REPO_DIR = '/content/ANT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaaninel/ANT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)

# HuggingFace login (set HF_TOKEN in Colab secrets)
from huggingface_hub import login, create_repo
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

HF_REPO = 'kaaninel/ANT'
try:
    create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'✓ HF repo: https://huggingface.co/{HF_REPO}')
except: pass

## 2. Configuration
Edit these values and re-run to adjust on the fly.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAINING CONFIG — Edit and re-run this cell to change params
# ═══════════════════════════════════════════════════════════════

# Steps per phase (total = sum of all three)
PHASE1_STEPS = 2000   # Phase 1: English LM only
PHASE2_STEPS = 2000   # Phase 2: LM + Memory QA
PHASE3_STEPS = 1000   # Phase 3: LM + QA + Chat

# Batch size: micro-batch × grad_accum = effective batch
# Sliding window creates B×T windows — large B explodes VRAM.
# Use small micro-batch with accumulation instead.
BATCH_SIZE = 32       # Micro-batch (per accumulation step)
GRAD_ACCUM = 8        # Accumulation steps → effective B=256

# Data sizes
N_WIKI = 50000        # Wikipedia sentences
N_SHELL = 5000        # Shell commands
N_QA = 30000          # QA training examples

# Eval frequency
EVAL_INTERVAL = 500

# Architecture
WINDOW_SIZE = 8
NUM_PASSES = 4

# A100 optimizations
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

total = PHASE1_STEPS + PHASE2_STEPS + PHASE3_STEPS
eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f'Total steps: {total}')
print(f'Micro-batch: {BATCH_SIZE} × {GRAD_ACCUM} accum = {eff_batch} effective')
print(f'Tokens/step: {eff_batch * 192:,}')
# A100 with B=32 sliding: ~10-15 it/s
print(f'Est. time: ~{total / 10:.0f}s = ~{total / 10 / 60:.1f} min (at ~10 it/s on A100)')


## 3. Train
Single command — all phases run automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRAIN — Direct Python call, no subprocess
# ═══════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F

from config import ModelConfig
from model import ANT
from train_micro import (
    VOCAB_SIZE, VOCAB, train_chat, count_params, log_model_summary,
    sliding_lm_encode,
)

device = 'cuda'

# Build model
cfg = ModelConfig()
cfg.vocab_size = VOCAB_SIZE
model = ANT(cfg, use_checkpoint=False).to(device)
log_model_summary(model, cfg)

print(f'Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')

best_acc = train_chat(
    model, cfg, device,
    output_dir='/content/ant_checkpoints/chat',
    steps_phase1=PHASE1_STEPS,
    steps_phase2=PHASE2_STEPS,
    steps_phase3=PHASE3_STEPS,
    lr=3e-4,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    eval_interval=EVAL_INTERVAL,
    window_size=WINDOW_SIZE,
    num_passes=NUM_PASSES,
    n_wiki=N_WIKI,
    n_shell=N_SHELL,
    n_qa=N_QA,
    n_chat=5000,
    use_bf16=True,
    use_compile=True,
    hf_repo='kaaninel/ANT',
)

print(f'\n✓ Training complete. Best QA: {best_acc:.1%}')


## 4. Test Generation
Quick test of the trained model.

In [ ]:
import torch, torch.nn.functional as F
from config import ModelConfig
from model import ANT
from train_micro import (
    VOCAB_SIZE, VOCAB, BOS_ID, tokenize, detokenize,
    sliding_generate, encode_sentence_frozen, sliding_lm_encode
)

# Load best checkpoint
import os, glob as gmod
ckpt_dir = '/content/ant_checkpoints/chat'
best = os.path.join(ckpt_dir, 'checkpoint_best.pt')
if not os.path.exists(best):
    best = os.path.join(ckpt_dir, 'checkpoint_latest.pt')

ckpt = torch.load(best, map_location='cuda', weights_only=False)
cfg = ModelConfig()
cfg.vocab_size = VOCAB_SIZE
model = ANT(cfg, use_checkpoint=False).cuda()
model.load_state_dict(ckpt['model'])
model.eval()

W = ckpt.get('window_size', 8)
P = ckpt.get('num_passes', 4)
print(f'Loaded step={ckpt["step"]}, QA={ckpt.get("qa_accuracy", 0):.1%}')

# Generate samples
prompts = ['The ', '## History\n\n', '$ ls -la', 'Hello, ']
for p in prompts:
    ids = [BOS_ID] + tokenize(p)
    out = sliding_generate(model, ids, max_new_tokens=100,
                           temperature=0.8, top_k=30,
                           window_size=W, num_passes=P)
    print(f'\n--- Prompt: {repr(p)} ---')
    print(detokenize(out)[:200])

## 5. Upload Final Checkpoint

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

ckpt_dir = '/content/ant_checkpoints/chat'
for f in ['checkpoint_latest.pt', 'checkpoint_best.pt']:
    path = os.path.join(ckpt_dir, f)
    if os.path.exists(path):
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f'checkpoints/chat/{f}',
            repo_id='kaaninel/ANT', repo_type='model')
        print(f'✓ Uploaded {f}')

print('\nDone! Check https://huggingface.co/kaaninel/ANT')